In [1]:
import setup
setup.init_django()

In [10]:
import requests

from datetime import datetime
from urllib.parse import urlencode

from django.conf import settings
from pytz import timezone

ALPHA_VANTAGE_API_KEY = settings.ALPHA_VANTAGE_API_KEY
assert ALPHA_VANTAGE_API_KEY != ""

In [3]:
symbol="AAPL"

In [4]:
month = datetime.now().strftime("%Y-%m")
month

'2024-10'

In [5]:
params = {
    "function": "TIME_SERIES_INTRADAY",
    "apikey": ALPHA_VANTAGE_API_KEY,
    "symbol": symbol,
    "interval": "1min",
    "extended_hours": "false",
    "outputsize": "full",
    "month": month,
}

In [6]:
params = urlencode(params)

In [8]:
url = f'https://www.alphavantage.co/query?{params}'
r = requests.get(url)
data = r.json()

In [9]:
data

{'Meta Data': {'1. Information': 'Intraday (1min) open, high, low, close prices and volume',
  '2. Symbol': 'AAPL',
  '3. Last Refreshed': '2024-10-14 15:59:00',
  '4. Interval': '1min',
  '5. Output Size': 'Full size',
  '6. Time Zone': 'US/Eastern'},
 'Time Series (1min)': {'2024-10-14 15:59:00': {'1. open': '231.4100',
   '2. high': '231.5000',
   '3. low': '231.2700',
   '4. close': '231.4000',
   '5. volume': '973063'},
  '2024-10-14 15:58:00': {'1. open': '231.4468',
   '2. high': '231.4800',
   '3. low': '231.3600',
   '4. close': '231.4000',
   '5. volume': '353910'},
  '2024-10-14 15:57:00': {'1. open': '231.4800',
   '2. high': '231.4900',
   '3. low': '231.3100',
   '4. close': '231.4450',
   '5. volume': '294687'},
  '2024-10-14 15:56:00': {'1. open': '231.3300',
   '2. high': '231.5000',
   '3. low': '231.2600',
   '4. close': '231.4750',
   '5. volume': '260450'},
  '2024-10-14 15:55:00': {'1. open': '231.1100',
   '2. high': '231.3600',
   '3. low': '231.0400',
   '4. cl

In [24]:
stock_market_data_mapping = {
    '1. open': 'open_price',
    '2. high': 'high',
    '3. low': 'low',
    '4. close': 'close_price', 
    '5. volume': 'volume'
}

def rename_sm_keys(og_dict):
    return {stock_market_data_mapping[k]: v for k,v in og_dict.items()}

In [28]:
tz = timezone("US/Eastern")

def timezone_as_datetime_obj(timezone_str, use_tz=True):
    as_datetime = datetime.strptime(timezone_str, "%Y-%m-%d %H:%M:%S")
    if not use_tz:
        return as_datetime
    return tz.localize(as_datetime)

In [32]:
dataset = data['Time Series (1min)']
timestamps = dataset.keys()

for timestamp in timestamps:
    sm_data_raw = dataset[timestamp]
    cleaned_data = rename_sm_keys(sm_data_raw)
    cleaned_data['time'] = timezone_as_datetime_obj(timestamp, use_tz=True)
    print(cleaned_data)

{'open_price': '231.4100', 'high': '231.5000', 'low': '231.2700', 'close_price': '231.4000', 'volume': '973063', 'time': datetime.datetime(2024, 10, 14, 15, 59, tzinfo=<DstTzInfo 'US/Eastern' EDT-1 day, 20:00:00 DST>)}
{'open_price': '231.4468', 'high': '231.4800', 'low': '231.3600', 'close_price': '231.4000', 'volume': '353910', 'time': datetime.datetime(2024, 10, 14, 15, 58, tzinfo=<DstTzInfo 'US/Eastern' EDT-1 day, 20:00:00 DST>)}
{'open_price': '231.4800', 'high': '231.4900', 'low': '231.3100', 'close_price': '231.4450', 'volume': '294687', 'time': datetime.datetime(2024, 10, 14, 15, 57, tzinfo=<DstTzInfo 'US/Eastern' EDT-1 day, 20:00:00 DST>)}
{'open_price': '231.3300', 'high': '231.5000', 'low': '231.2600', 'close_price': '231.4750', 'volume': '260450', 'time': datetime.datetime(2024, 10, 14, 15, 56, tzinfo=<DstTzInfo 'US/Eastern' EDT-1 day, 20:00:00 DST>)}
{'open_price': '231.1100', 'high': '231.3600', 'low': '231.0400', 'close_price': '231.3300', 'volume': '291157', 'time': dat